In lecture 25 of MCS 320, we consider symbolic-numeric computing

# 1. Interval Arithmetic

What is the right precision to obtain accurate results?

To motivate the use of interval arithmetic, we take an example
from the paper of Stefano Taschini in the proceedings of SciPy 2008.
We convert the floating-point coefficients to rational numbers.

In [1]:
F(x,y) = (QQ(333.75) - x**2)*y**6 + x**2*(11*x**2*y**2 - 121*y**4 - 2) + QQ(5.5)*y**8 + x/(2*y)
F

(x, y) |--> 11/2*y^8 - 1/4*(4*x^2 - 1335)*y^6 + (11*x^2*y^2 - 121*y^4 - 2)*x^2 + 1/2*x/y

In [2]:
(A, B) = (77617, 33096)
exact = F(A, B)
exact

-54767/66192

In [3]:
exact.n(digits=10)

-0.8273960599

In [4]:
[F(A.n(digits=d), B.n(digits=d)) for d in range(10, 40)]

[-6.662869420e25,
 1.0742565174e25,
 -1.37766841984e23,
 1.334889447551e22,
 3.9041564432186e21,
 -8.18209863729137e20,
 -6.553155272242627e18,
 -6.5531536229751849e18,
 3.64375473385373696e17,
 4.087503195734016000e15,
 4.0875015851212800000e15,
 1.46852011835392000000e14,
 6.114519285760000000000e12,
 6.1145192857600000000000e12,
 6.72051363840000000000000e10,
 -1.514340352000000000000000e9,
 -1.5143403520000000000000000e9,
 9.62723840000000000000000000e7,
 -3.794534400000000000000000000e7,
 3.9976940000000000000000000000e6,
 -196610.000000000000000000000000,
 -2.000000000000000000000000000000,
 -2.0000000000000000000000000000000,
 -2.00000000000000000000000000000000,
 -2.000000000000000000000000000000000,
 -2.0000000000000000000000000000000000,
 -1.00000000000000000000000000000000000,
 -0.8125000000000000000000000000000000000,
 -0.82812500000000000000000000000000000000,
 -0.827148437500000000000000000000000000000]

We see that we eventually get to an accurate result.  However, observe the plateaus which could give the impression that, as the result no longer changes, we have reached an accurate result.

In [5]:
RIF

Real Interval Field with 53 bits of precision

In [6]:
y = F(RIF(A), RIF(B))
y

0.?e22

To see better the range of numbers, we ask to see the number in the ``brackets`` style.

In [7]:
RIF(y).str(style='brackets')

'[-3.1793933514545646e21 .. 3.9041567246936242e21]'

Observe the magnitude of the numbers.  The range of the interval is huge!

In [8]:
R100 = RealIntervalField(prec=100)
R100

Real Interval Field with 100 bits of precision

In [9]:
y100 = F(R100(A), R100(B))
R100(y100).str(style='brackets')

'[-4.3909140000000000000000000000000e6 .. 1.2386302000000000000000000000000e7]'

After doubling the precision to 100 bits, observe that the magnitude dropped from $10^{21}$ down to $10^7$.

In [10]:
R200 = RealIntervalField(prec=200)
y200 = F(R200(A), R200(B))
R200(y200).str(style='brackets')

'[-0.82739605994682136814116587127405770020482922433302519493736327 .. -0.82739605994682136814115925382915727598343025306348863523453474]'

Working with 200 bits gives an accurate result.  The side of the interval gives a bound on the error.

In [11]:
y200

-0.82739605994682136814116?

In [12]:
ry200 = R200(y200)
ry200.upper() - ry200.lower()

6.6174449004242213989712695365597028285264968872070312500000e-24

The error is defined as the difference between the upper and lower limit of the interval and we see that we have about 24 correct decimal places.

# 2. Symbolic-Numeric Factorization

In a numeric factorization of a polynomial in one variable, we compute the complex roots of the polynomial.  In a symbolic factorization, we extend the field of rational numbers with algebraic number till the polynomial factors as a product of linear forms.

A symbolic-numeric factorization works over the ``QQbar`` number field, which uses the complex interval field.

In [13]:
P.<x> = PolynomialRing(QQbar)
P

Univariate Polynomial Ring in x over Algebraic Field

In [14]:
q = x^3 + x + 1
f = factor(q)
f

(x - 0.3411639019140097? - 1.161541399997252?*I) * (x - 0.3411639019140097? + 1.161541399997252?*I) * (x + 0.6823278038280193?)

In [15]:
q.roots(QQbar)

[(-0.6823278038280193?, 1),
 (0.3411639019140097? - 1.161541399997252?*I, 1),
 (0.3411639019140097? + 1.161541399997252?*I, 1)]

If we want a more accurate symbolic-numeric factorization, then we can compute the roots at a higher precision, for example with 200 bits.

In [16]:
C = ComplexIntervalField(prec=200)
C

Complex Interval Field with 200 bits of precision

In [17]:
q.roots(C)

[(-0.682327803828019327369483739711048256891188581897998577803728?, 1),
 (0.34116390191400966368474186985552412844559429094899928890187? - 1.16154139999725193608791768724717407484314725802151429063617?*I,
  1),
 (0.34116390191400966368474186985552412844559429094899928890187? + 1.16154139999725193608791768724717407484314725802151429063617?*I,
  1)]

# 3. Constrained Optimization

To end the ``calculus`` chapter and to transition into the solving chapter, consider a problem of constrained optimization.

In [18]:
x, y, z = var('x, y, z')
target = x^2 + 2*x*y*z - z^2
constraint = x^2 + y^2 + z^2 - 1 == 0
print('optimize', target, 'subject to', constraint)

optimize 2*x*y*z + x^2 - z^2 subject to x^2 + y^2 + z^2 - 1 == 0


Let us make plot, for the value of the target equal to two.

In [19]:
tplot = implicit_plot3d(target-2, (x,-2,2), (y,-2,2), (z,-2,2), color='red')
cplot = implicit_plot3d(constraint.lhs(), (x,-2,2), (y,-2,2), (z,-2,2))
show(tplot+cplot)

Graphics3d Object

If the value of the target equals one, then we see that the red sheets touch the blue ball.

In [20]:
tplot = implicit_plot3d(target-1, (x,-2,2), (y,-2,2), (z,-2,2), color='red')
cplot = implicit_plot3d(constraint.lhs(), (x,-2,2), (y,-2,2), (z,-2,2))
show(tplot+cplot)

Graphics3d Object

To apply the method of Lagrange multipliers we need the gradients.

In [21]:
ft(x,y,z) = target
fc(x,y,z) = constraint.lhs()
print('target as function :', ft)
print('constraint as function :', fc)

target as function : (x, y, z) |--> 2*x*y*z + x^2 - z^2
constraint as function : (x, y, z) |--> x^2 + y^2 + z^2 - 1


In [22]:
gt = ft.diff()
gt

(x, y, z) |--> (2*y*z + 2*x, 2*x*z, 2*x*y - 2*z)

In [23]:
gc = fc.diff()
gc

(x, y, z) |--> (2*x, 2*y, 2*z)

Now we set up the system.  Because ``lambda`` is reserved in Python, using ``lambda`` as the name of a variable is wrong.

In [24]:
w = var('w')
eqs = [gt(x,y,z)[k] - w*gc(x,y,z)[k] == 0 for k in range(3)]
eqs.append(constraint)
eqs

[-2*w*x + 2*y*z + 2*x == 0,
 -2*w*y + 2*x*z == 0,
 2*x*y - 2*w*z - 2*z == 0,
 x^2 + y^2 + z^2 - 1 == 0]

Observe that the constraint was appended to the list of equations.  We then just solve.

In [25]:
sols = solve(eqs,[x,y,z,w])
sols

[[x == 1, y == 0, z == 0, w == 1], [x == -1, y == 0, z == 0, w == 1], [x == 0, y == 0, z == -1, w == -1], [x == 0, y == 0, z == 1, w == -1], [x == 0, y == -1, z == 0, w == 0], [x == 0, y == 1, z == 0, w == 0]]

To evaluate the solutions at the target, we must convert the coordinates of the solutions into tuples and drop the value for w.

In [26]:
[sol for sol in sols]

[[x == 1, y == 0, z == 0, w == 1],
 [x == -1, y == 0, z == 0, w == 1],
 [x == 0, y == 0, z == -1, w == -1],
 [x == 0, y == 0, z == 1, w == -1],
 [x == 0, y == -1, z == 0, w == 0],
 [x == 0, y == 1, z == 0, w == 0]]

In [27]:
L = [[e.rhs() for e in sol] for sol in sols]
L

[[1, 0, 0, 1],
 [-1, 0, 0, 1],
 [0, 0, -1, -1],
 [0, 0, 1, -1],
 [0, -1, 0, 0],
 [0, 1, 0, 0]]

In [28]:
T = [tuple(e[0:3]) for e in L]
T

[(1, 0, 0), (-1, 0, 0), (0, 0, -1), (0, 0, 1), (0, -1, 0), (0, 1, 0)]

In [29]:
[ft(x,y,z) for (x,y,z) in T]

[1, 1, -1, -1, 0, 0]

This problem has two maxima, valued at 1, and two minima, with value -1.